In [12]:
import pandas as pd

In [13]:
fact_re_state_2025 = pd.read_csv("../clean/fact_re_state_2025.csv")
fact_re_state_potential = pd.read_csv("../clean/fact_re_state_potential.csv")
fact_re_capcity_long = pd.read_csv("../clean/fact_re_capacity_long.csv")

Merge the three cleaned datasets and add new features

In [14]:
mnre_features = fact_re_state_2025[
    [
        "state_id",
        "state_name",
        "Small_Hydro_MW",
        "Wind_Power_MW",
        "Bio_Power_MW",
        "Solar_Power_MW",
        "Large_Hydro_MW",
        "Total_RES_MW"
    ]
].copy()

In [15]:
mnre_features = mnre_features.merge(
    fact_re_state_potential[
        [
            "state_id",
            "total_re_potential_mw"
        ]
    ],
    on="state_id",
    how="left",
    validate="one_to_one"
)

In [16]:
mnre_features["renewable_headroom_mw"] = (
    mnre_features["total_re_potential_mw"]
    - mnre_features["Total_RES_MW"]
)

In [17]:
mnre_features["renewable_utilization_pct"] = (
    mnre_features["Total_RES_MW"]
    / mnre_features["total_re_potential_mw"]
    * 100
)

In [19]:
history_pivot = fact_re_capcity_long.pivot(
    index="state_id",
    columns="fiscal_year",
    values="cumulative_re_capacity_mw"
).reset_index()

In [20]:
history_pivot = history_pivot.rename(columns={
    "2017_18": "re_capacity_2017_18_mw",
    "2024_25": "re_capacity_2024_25_mw"
})

In [21]:
mnre_features = mnre_features.merge(
    history_pivot[
        [
            "state_id",
            "re_capacity_2017_18_mw",
            "re_capacity_2024_25_mw"
        ]
    ],
    on="state_id",
    how="left",
    validate="one_to_one"
)

In [22]:
mnre_features["re_capacity_growth_mw"] = (
    mnre_features["re_capacity_2024_25_mw"]
    - mnre_features["re_capacity_2017_18_mw"]
)

In [23]:
mnre_features["re_capacity_growth_pct"] = (
    mnre_features["re_capacity_growth_mw"]
    / mnre_features["re_capacity_2017_18_mw"]
    * 100
)

In [24]:
mnre_features["re_capacity_cagr_pct"] = (
    (
        mnre_features["re_capacity_2024_25_mw"]
        / mnre_features["re_capacity_2017_18_mw"]
    ) ** (1 / 7)
    - 1
) * 100

In [25]:
print(mnre_features.shape)
print(mnre_features.columns.tolist())
print(mnre_features.isna().sum())
print(
    mnre_features[
        [
            "state_name",
            "Total_RES_MW",
            "total_re_potential_mw",
            "renewable_headroom_mw",
            "renewable_utilization_pct",
            "re_capacity_growth_mw",
            "re_capacity_growth_pct",
            "re_capacity_cagr_pct"
        ]
    ].to_string(index=False)
)

(36, 16)
['state_id', 'state_name', 'Small_Hydro_MW', 'Wind_Power_MW', 'Bio_Power_MW', 'Solar_Power_MW', 'Large_Hydro_MW', 'Total_RES_MW', 'total_re_potential_mw', 'renewable_headroom_mw', 'renewable_utilization_pct', 're_capacity_2017_18_mw', 're_capacity_2024_25_mw', 're_capacity_growth_mw', 're_capacity_growth_pct', 're_capacity_cagr_pct']
state_id                     0
state_name                   0
Small_Hydro_MW               0
Wind_Power_MW                0
Bio_Power_MW                 0
Solar_Power_MW               0
Large_Hydro_MW               0
Total_RES_MW                 0
total_re_potential_mw        0
renewable_headroom_mw        0
renewable_utilization_pct    0
re_capacity_2017_18_mw       1
re_capacity_2024_25_mw       1
re_capacity_growth_mw        1
re_capacity_growth_pct       1
re_capacity_cagr_pct         1
dtype: int64
                              state_name  Total_RES_MW  total_re_potential_mw  renewable_headroom_mw  renewable_utilization_pct  re_capacity_growt

In [26]:
print(
    fact_re_state_potential[
        fact_re_state_potential["state_name"].isin([
            "Chandigarh",
            "Ladakh"
        ])
    ].to_string(index=False)
)

state_id state_name  wind_potential_mw  small_hydro_potential_mw  biomass_potential_mw  bagasse_cogen_potential_mw  solar_ground_potential_mw  large_hydro_potential_mw  total_re_potential_mw
   IN-LA     Ladakh                1.0                       0.0                  0.00                         0.0                    8556.64                       0.0                8557.64
   IN-CH Chandigarh                0.0                       0.0                  0.15                         0.0                      22.42                       0.0                  22.57


In [28]:
import numpy as np

In [29]:
mnre_features[
    ["re_capacity_growth_pct", "re_capacity_cagr_pct"]
] = mnre_features[
    ["re_capacity_growth_pct", "re_capacity_cagr_pct"]
].replace([np.inf, -np.inf], np.nan)

In [30]:
mnre_features = mnre_features.rename(columns={
    "Small_Hydro_MW": "small_hydro_mw",
    "Wind_Power_MW": "wind_power_mw",
    "Bio_Power_MW": "bio_power_mw",
    "Solar_Power_MW": "solar_power_mw",
    "Large_Hydro_MW": "large_hydro_mw",
    "Total_RES_MW": "total_re_mw"
})

In [31]:
print(mnre_features.shape)
print(mnre_features.columns.tolist())
print(mnre_features.isna().sum())

(36, 16)
['state_id', 'state_name', 'small_hydro_mw', 'wind_power_mw', 'bio_power_mw', 'solar_power_mw', 'large_hydro_mw', 'total_re_mw', 'total_re_potential_mw', 'renewable_headroom_mw', 'renewable_utilization_pct', 're_capacity_2017_18_mw', 're_capacity_2024_25_mw', 're_capacity_growth_mw', 're_capacity_growth_pct', 're_capacity_cagr_pct']
state_id                     0
state_name                   0
small_hydro_mw               0
wind_power_mw                0
bio_power_mw                 0
solar_power_mw               0
large_hydro_mw               0
total_re_mw                  0
total_re_potential_mw        0
renewable_headroom_mw        0
renewable_utilization_pct    0
re_capacity_2017_18_mw       1
re_capacity_2024_25_mw       1
re_capacity_growth_mw        1
re_capacity_growth_pct       2
re_capacity_cagr_pct         2
dtype: int64


In [32]:
mnre_features.to_csv(
    "../clean/mnre_features.csv",
    index=False
)